# Regression Model Comparison

This notebook compares a mean baseline, Ridge Regression, and a one-hidden-layer
artificial neural network (ANN). It uses the original 10-by-10 nested
cross-validation design for hyperparameter selection and outer-fold evaluation,
then applies the correlated t-test from the course material.

Run the cells from top to bottom. The saved outputs are from the original
project run; the models are not rerun during repository cleanup.


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler


## Dataset and feature definitions


In [2]:
DATA_PATH = Path("../data/Concrete_Data.xls")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Concrete_Data.xls was not found. Download the UCI dataset and place "
        "the file at data/Concrete_Data.xls."
    )

df = pd.read_excel(DATA_PATH)

new_column_names = [
    "Cement",
    "BlastFurnaceSlag",
    "FlyAsh",
    "Water",
    "Superplasticizer",
    "CoarseAggregate",
    "FineAggregate",
    "Age",
    "CompressiveStrength",
]

df.columns = new_column_names
print(df.head())

target_column = "CompressiveStrength"
X = df.drop(columns=[target_column])
y = df[target_column].to_frame()


   Cement  BlastFurnaceSlag  FlyAsh  Water  Superplasticizer  CoarseAggregate  \
0   540.0               0.0     0.0  162.0               2.5           1040.0   
1   540.0               0.0     0.0  162.0               2.5           1055.0   
2   332.5             142.5     0.0  228.0               0.0            932.0   
3   332.5             142.5     0.0  228.0               0.0            932.0   
4   198.6             132.4     0.0  192.0               0.0            978.4   

   FineAggregate  Age  CompressiveStrength  
0          676.0   28            79.986111  
1          676.0   28            61.887366  
2          594.0  270            40.269535  
3          594.0  365            41.052780  
4          825.5  360            44.296075  


## Candidate models and hyperparameters


In [3]:
h_range = [20, 50, 80, 110, 140]
lambda_candidates = np.logspace(-3, 3, 20)

# Baseline model function

def get_baseline_predictions(y_train, y_test):
    y_train_mean = np.mean(y_train.values)
    y_pred_baseline = np.full(shape=y_test.shape, fill_value=y_train_mean)
    return y_pred_baseline

## Nested cross-validation


In [4]:
K1 = 10  # Outer folds (generalization estimate)
K2 = 10  # Inner folds (model selection)
RANDOM_STATE = 42

CV_outer = KFold(n_splits=K1, shuffle=True, random_state=RANDOM_STATE)


results = {
    'fold': [],
    'best_h': [], # ANN için en iyi h
    'best_lambda': [], # Lineer Regresyon için en iyi lambda
    'ANN_E_test': [],
    'LR_E_test': [], # Düzenlileştirilmiş Lineer Regresyon (LR)
    'Baseline_E_test': []
}

# ----------------------------------------------------------------------------------
# ANA DÖNGÜ BAŞLANGICI: Dış Katman (K1)
# ----------------------------------------------------------------------------------
for i, (train_par_index, test_index) in enumerate(CV_outer.split(X)): # cv_outer.split(x)-> train_par_index , test_index (data setindeki satır konumları) üretir. Toplam 1030 satır var data setinde.  103 satır konumu (%10) test_index, 927 satır konumu train_par_index olur.
#train_par_iindex ve test_index satır konumlarıdır. mesela 1. satır, 5. satır gibi. 
    print(f"Outer Fold: {i+1}/{K1}")

    # A. Veri Bölme (Dış Katman)
    X_test = X.iloc[test_index, :] #iloc ile satır konumlarındaki verileri alıyoruz.
    y_test = y.iloc[test_index]
    
    X_par = X.iloc[train_par_index, :] # D_i^par
    y_par = y.iloc[train_par_index] # D_i^par


    # B. Temel Modelin (Baseline) Dış Test Hata Tahmini
    # Baseline modeli eğitim/parametre seçimi verisi üzerinde 'eğitilir' (ortalaması bulunur)
    # ve dış test setinde değerlendirilir.
    y_pred_baseline = get_baseline_predictions(y_par, y_test)
    E_test_baseline = mean_squared_error(y_test, y_pred_baseline)

# ------------------------------------------------------------------------------
    # İÇ DÖNGÜ BAŞLANGICI (K2): En İyi Parametreleri (h* ve lambda*) Bulma
    # ------------------------------------------------------------------------------
    
    CV_inner = KFold(n_splits=K2, shuffle=True, random_state=RANDOM_STATE)
    
    # İç döngüde en iyi parametreyi bulmak için geçici değişkenler
    best_h = None
    min_ann_error = np.inf
    best_lambda = None
    min_lr_error = np.inf
    
    # İç döngüdeki her modelin hata sonuçlarını tutmak için dict'ler
    ann_h_errors = {h: [] for h in h_range}
    lr_lambda_errors = {l: [] for l in lambda_candidates}

    for j, (train_index, test_inner_index) in enumerate(CV_inner.split(X_par)):
        
        # 1. Veri Bölme (İç Katman)
        X_train = X_par.iloc[train_index, :] 
        X_test_inner = X_par.iloc[test_inner_index, :]

        y_train = y_par.iloc[train_index]
        y_test_inner = y_par.iloc[test_inner_index]

        # 2. Standardizasyon (KRİTİK ADIM)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)  #inner train setinde her bir özellik sütunun ortlama ve stardart sapmasını öğrenir. öğrendiklerini kullanarak dönüştürür (her sütunun ort 0 ve standart sapması 1 olcak şekilde).
        X_test_inner_scaled = scaler.transform(X_test_inner)    # x_trainde öğrenilen ort ve std sapma değerlerini kullanarak x_test_inner'ı dönüştürür.
        
        # y'yi NumPy array'e dönüştürelim
        y_train_np = y_train.values.ravel()     #ravel() 2D array'i 1D array'e çevirir.
        y_test_inner_np = y_test_inner.values.ravel()


        # 3. Model Seçimi: Tüm h ve lambda'ları Dene
        
        # ANN Modeli Denemeleri
        for h in h_range:
            ann_model = MLPRegressor(hidden_layer_sizes=(h,), 
                                     solver='lbfgs',
                                     activation='relu',
                                     max_iter=3000, 
                                     random_state=RANDOM_STATE)
            ann_model.fit(X_train_scaled, y_train_np)
            y_pred_ann = ann_model.predict(X_test_inner_scaled)
            mse = mean_squared_error(y_test_inner_np, y_pred_ann)
            ann_h_errors[h].append(mse)

        # Düzenlileştirilmiş Lineer Regresyon (Ridge) Denemeleri
        for l in lambda_candidates:
            # Ridge modelinde lambda parametresi 'alpha' olarak geçer
            lr_model = Ridge(alpha=l) 
            lr_model.fit(X_train_scaled, y_train_np)
            y_pred_lr = lr_model.predict(X_test_inner_scaled)
            mse = mean_squared_error(y_test_inner_np, y_pred_lr)
            lr_lambda_errors[l].append(mse)

    # İç Döngü Sonlanır. Optimal Parametreler Seçilir.
    
    # En iyi h* (ANN)
    avg_ann_errors = {h: np.mean(errors) for h, errors in ann_h_errors.items()}
    best_h = min(avg_ann_errors, key=avg_ann_errors.get)
    
    # En iyi lambda* (Lineer Regresyon)
    avg_lr_errors = {l: np.mean(errors) for l, errors in lr_lambda_errors.items()}
    best_lambda = min(avg_lr_errors, key=avg_lr_errors.get)
    
    print(f"  -> Best Parameters: h*={best_h}, lambda*={best_lambda}")
    
    # ------------------------------------------------------------------------------
    # DIŞ TEST DEĞERLENDİRMESİ
    # ------------------------------------------------------------------------------
    
    # 1. Modelleri Optimal Parametrelerle, D_i^par verisinin tamamı üzerinde EĞİT
    
    # X_par'ı D_i^par'ın tamamı üzerinde standardize et
    scaler_final = StandardScaler()
    X_par_scaled = scaler_final.fit_transform(X_par)
    X_test_scaled = scaler_final.transform(X_test)
    # y'yi NumPy array'e dönüştür
    y_par_np = y_par.values.ravel()
    y_test_np = y_test.values.ravel()
    
    # 2. ANN'i en iyi h* ile EĞİT
    ann_model_final = MLPRegressor(hidden_layer_sizes=(best_h,), 
                                   solver='lbfgs', 
                                   activation='relu',
                                   max_iter=3000, 
                                   random_state=RANDOM_STATE)
    ann_model_final.fit(X_par_scaled, y_par_np)
    y_pred_ann_final = ann_model_final.predict(X_test_scaled)
    E_test_ann = mean_squared_error(y_test_np, y_pred_ann_final)
    
    # 3. Lineer Regresyonu en iyi lambda* ile EĞİT
    lr_model_final = Ridge(alpha=best_lambda)
    lr_model_final.fit(X_par_scaled, y_par_np)
    y_pred_lr_final = lr_model_final.predict(X_test_scaled)
    E_test_lr = mean_squared_error(y_test_np, y_pred_lr_final)
    
    
    # 4. Sonuçları Topla
    results['fold'].append(i + 1)
    results['best_h'].append(best_h)
    results['best_lambda'].append(best_lambda)
    results['ANN_E_test'].append(E_test_ann)
    results['LR_E_test'].append(E_test_lr)
    results['Baseline_E_test'].append(E_test_baseline)

# ----------------------------------------------------------------------------------
# DÖNGÜ SONU: Sonuç Tablosunu Göster
# ----------------------------------------------------------------------------------
final_df = pd.DataFrame(results)
print("\n--- Two Level Cross Validation Results Table ---")
print(final_df)

print("\n--- Mean Test Results ---")
print(f"ANN mean MSE: {final_df['ANN_E_test'].mean():.4f}")
print(f"LR mean MSE: {final_df['LR_E_test'].mean():.4f}")
print(f"Baseline mean MSE: {final_df['Baseline_E_test'].mean():.4f}")


Outer Fold: 1/10


d:\Anaconda\envs\dtu_master\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
d:\Anaconda\envs\dtu_master\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
d:\Anaconda\envs\d

  -> Best Parameters: h*=80, lambda*=1.438449888287663


d:\Anaconda\envs\dtu_master\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Outer Fold: 2/10


...

d:\Anaconda\envs\dtu_master\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Outer Fold: 10/10


d:\Anaconda\envs\dtu_master\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
d:\Anaconda\envs\dtu_master\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
d:\Anaconda\envs\d

  -> Best Parameters: h*=50, lambda*=0.001

--- Two Level Cross Validation Results Table ---
   fold  best_h  best_lambda  ANN_E_test   LR_E_test  Baseline_E_test
0     1      80     1.438450   27.941486  101.221629       268.475053
1     2      50     1.438450   28.286942   86.281129       247.717454
2     3      80     0.695193   20.336760  135.570794       299.112382
3     4      80     1.438450   37.639968  133.604147       371.855304
4     5     140     1.438450   40.595865  142.398140       263.154527
5     6     110     1.438450   11.662361  108.839689       304.370672
6     7      80     2.976351   28.856503   90.181676       302.581038
7     8      80     0.335982   19.511384   89.336558       261.798622
8     9      50     0.162378   29.618045  105.182437       244.908005
9    10      50     0.001000   21.984779  105.700448       227.350799

--- Mean Test Results ---
ANN mean MSE: 26.6434
LR mean MSE: 109.8317
Baseline mean MSE: 279.1324



d:\Anaconda\envs\dtu_master\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:602: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


## Correlated statistical comparison

The test uses the outer-fold errors already stored in `final_df`. No
intermediate CSV file is required.


In [6]:
# Correlated t-test (Setup II)
def correlated_ttest(r, K, alpha=0.05):
    """
    Calculate the correlated t-test for Setup II.
    r: Array of error differences (E_A - E_B)
    K: Number of outer folds (10)
    """
    r = np.array(r)
    J = len(r)          # Number of folds (10)
    r_hat = np.mean(r)  # Mean error difference
    s_hat = np.std(r, ddof=1) # Sample standard deviation

    # Correlation coefficient (rho = 1/K)
    rho = 1 / K

    # Correlation-adjusted standard error
    # Formül: s_hat * sqrt( (1/J) + (rho / (1 - rho)) )
    sigma_tilde = s_hat * np.sqrt((1 / J) + (rho / (1 - rho)))

    # Confidence interval
    CI = st.t.interval(1 - alpha, df=J - 1, loc=r_hat, scale=sigma_tilde)

    # Two-sided p-value
    # H0: the mean difference is zero
    p = 2 * st.t.cdf(-np.abs(r_hat) / sigma_tilde, df=J - 1)

    return r_hat, CI, p


In [8]:
# Number of outer folds
K = 10 
ALPHA = 0.05

# Reuse the outer-fold results already held in memory.
df_errors = final_df.copy()

# Extract model errors
E_ANN = df_errors['ANN_E_test']
E_LR = df_errors['LR_E_test']
E_BASELINE = df_errors['Baseline_E_test']

print("--- STATISTICAL COMPARISON RESULTS (Setup II) ---")
print(f"Significance Level (Alpha): {ALPHA}\n")

# --- COMPARISON 1: LR vs. ANN ---
# r_i = E_LR - E_ANN (If positive, LR is worse, ANN is better)
r_LR_vs_ANN = E_LR - E_ANN
r_hat, ci, p = correlated_ttest(r_LR_vs_ANN, K, ALPHA)

print("1. Linear Regression (LR) vs. ANN")
print(f"  Mean Difference (LR - ANN): {r_hat:.4f}")
print(f"  95% Confidence Interval (CI): [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"  p-value: {p:.8f}")
print(f"  Conclusion: ANN is {'STATISTICALLY SIGNIFICANTLY' if p < ALPHA else 'not statistically'} better than LR.\n")


# --- COMPARISON 2: ANN vs. BASELINE ---
# r_i = E_BASELINE - E_ANN (If positive, Baseline is worse, ANN is better)
r_ANN_vs_B = E_BASELINE - E_ANN
r_hat, ci, p = correlated_ttest(r_ANN_vs_B, K, ALPHA)

print("2. ANN vs. BASELINE")
print(f"  Mean Difference (Baseline - ANN): {r_hat:.4f}")
print(f"  95% Confidence Interval (CI): [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"  p-value: {p:.8f}")
print(f"  Conclusion: ANN is {'STATISTICALLY SIGNIFICANTLY' if p < ALPHA else 'not statistically'} better than BASELINE.\n")


# --- COMPARISON 3: LR vs. BASELINE ---
# r_i = E_BASELINE - E_LR (If positive, Baseline is worse, LR is better)
r_LR_vs_B = E_BASELINE - E_LR
r_hat, ci, p = correlated_ttest(r_LR_vs_B, K, ALPHA)

print("3. LR vs. BASELINE")
print(f"  Mean Difference (Baseline - LR): {r_hat:.4f}")
print(f"  95% Confidence Interval (CI): [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"  p-value: {p:.8f}")
print(f"  Conclusion: LR is {'STATISTICALLY SIGNIFICANTLY' if p < ALPHA else 'not statistically'} better than BASELINE.\n")


--- STATISTICAL COMPARISON RESULTS (Setup II) ---
Significance Level (Alpha): 0.05

1. Linear Regression (LR) vs. ANN
  Mean Difference (LR - ANN): 83.1883
  95% Confidence Interval (CI): [63.6500, 102.7265]
  p-value: 0.00000489
  Conclusion: ANN is STATISTICALLY SIGNIFICANTLY better than LR.

2. ANN vs. BASELINE
  Mean Difference (Baseline - ANN): 252.4890
  95% Confidence Interval (CI): [209.6636, 295.3143]
  p-value: 0.00000031
  Conclusion: ANN is STATISTICALLY SIGNIFICANTLY better than BASELINE.

3. LR vs. BASELINE
  Mean Difference (Baseline - LR): 169.3007
  95% Confidence Interval (CI): [129.9338, 208.6677]
  p-value: 0.00000450
  Conclusion: LR is STATISTICALLY SIGNIFICANTLY better than BASELINE.

